# Lab 4 — Add a Multimodal Caption Stage to the Catalog Pipeline (AI-assisted)

**Time:** ~15–20 min  
**Mode:** AI-assisted (Claude Code or similar). The point is *not* to type the code — 
the point is to design the prompts, read the output critically, and verify it runs.

## Where this picks up
In the lecture you saw the `AddEmbedding` actor class wired into a Ray Data
pipeline that produces `desc_emb` for every product. In this lab you'll add a
second, *multimodal* stage that generates an extended product blurb (the same
shape as the `cat_desc` field produced by the lecture's `IT2T` actor and the
`create_extended_desc_job.py` script).

## Learning objectives
1. Drive an AI assistant to convert a non-Ray, single-record "hello world" into a
   Ray Data `map_batches` actor stage with the right resource flags.
2. Reason about `batch_size`, `compute=ActorPoolStrategy(...)`, `num_gpus`,
   and `repartition` together.
3. Catch common AI-generated bugs: missing `num_gpus`, batch-vs-row confusion,
   model loaded once per call instead of once per actor.

## Setup

In [ ]:
import ray, os

if not ray.is_initialized():
    ray.init()

INPUT_DIR = '/mnt/cluster_storage/cat_with_embeddings/'   # produced earlier in the lecture
TINY_DIR  = '/mnt/cluster_storage/scratch/tiny_sample/'
OUTPUT    = '/mnt/cluster_storage/scratch/lab_2_2_output/'
IMAGE_DIR = '/mnt/cluster_storage/catalog_images/'

# Make a *tiny* sample so the lab runs quickly while you iterate.
ray.data.read_parquet(INPUT_DIR).limit(16).write_parquet(TINY_DIR, mode=ray.data.SaveMode.OVERWRITE)
ray.data.read_parquet(TINY_DIR).schema()

## The "hello world" you'll convert
Below is a single-record example using a HuggingFace image-text-to-text
pipeline. This is the kind of code a teammate might paste from a model card.
It is **not** Ray Data — it loads the model every call, processes one row,
and uses no resource hints.

In [1]:
from transformers import pipeline
from PIL import Image

MODEL_NAME = '/mnt/cluster_storage/hf_cache/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767'

system_prompt = (
    'You are a helpful assistant. Given a product name and description, and an image of\n'
    'that product, please create a more descriptive and attractive blurb for the product,\n'
    'capturing elements from the image and suitable for use in an ecommerce website.\n'
    'Output a single suggestion only, with no extra conversational language.'
)

def hello_world_caption(item_id: str, desc: str) -> str:
    pipe = pipeline('image-text-to-text', model=MODEL_NAME, device='cuda:0')
    image_path = os.path.join(IMAGE_DIR, f'{item_id}.png')
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': system_prompt}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'path': image_path},
            {'type': 'text',  'text': desc},
        ]},
    ]
    out = pipe(text=[messages], max_new_tokens=256, batch_size=1)
    return out[0][0]['generated_text'][-1]['content']

## Exercise — drive the AI assistant through the conversion

### Prompt A — convert hello-world to a Ray Data actor stage

> *"Convert `hello_world_caption` into a Ray Data `map_batches` stage. Use a
> stateful actor class so the model is loaded **once per replica**, not once
> per call. The pipeline should read from `TINY_DIR`, produce a new column
> `cat_desc` per row, and write to `OUTPUT`. Use `compute=ActorPoolStrategy(...)`
> sized to all available GPUs in the cluster, `num_gpus=1` per replica, and
> `batch_size=8`. Repartition the input so that the actor pool has work to do."*

**Things to verify in the AI's output before running:**
1. Model is constructed in `__init__`, **not** `__call__`.
2. `num_gpus=1` appears on the `map_batches` call (not just inside the class).
3. `compute=ray.data.ActorPoolStrategy(size=int(ray.cluster_resources()['GPU']))`
   (or equivalent), evaluated dynamically — not hard-coded to 1.
4. The function builds **batched** `messages` and passes them all to `pipe(...)`
   in a single call (otherwise you lose the batching speedup).
5. The image path is constructed per row using the same `IMAGE_DIR` convention.

### Prompt B — sanity-check the output

> *"Read back from `OUTPUT` and show me 3 rows with the `desc` and the new
> `cat_desc`. Then write a short check that asserts every `cat_desc` is a
> non-empty string and is at least as long as the original `desc`."*

### Prompt C — push toward production

> *"What would I need to change to run this as an Anyscale Job over the
> *full* `INPUT_DIR` instead of `TINY_DIR`? List concrete changes only — 
> file paths, storage tier, repartition value, anything else."*

Compare the answer against `create_extended_desc_job.py` in the course repo.
(`/mnt/user_storage` instead of `/mnt/cluster_storage`, `repartition(2*GPUs)`,
removing the `limit`, etc.)

In [ ]:
# Paste / iterate on the AI-generated pipeline here.
